# Anomaly Detector: Z-score Debug & Threshold Tuning

This notebook is a **debug + tuning guide for the Isolation Forest anomaly detector**.

It is meant to:
- Confirm that z-score baselines are actually being used in anomaly alerts.
- Check that boosted anomaly scores behave as expected.
- Explore thresholds that **reduce anomaly spam** while still lighting up on known Scenario-1 insiders.

---

## How to use this notebook (read me first)

This notebook is the **gatekeeper** before you run a full multi-month `run_loop`.

Follow this order **exactly**:

1. **Do not touch `run_loop` until the smoke test passes.**  
   Run:

       python -m src.anomaly.smoke_test_anomaly_zscores

   If this fails, **stop**. Fix anomaly artifacts first.

2. **Run a short preflight `run_loop`, NOT the full company timeline.**  
   Example:

       python -m src.run_loop --start 2010-08-20 --end 2010-08-30 --out-dir out/r5.2/alerts_preflight

   This small window is where we validate anomaly behavior without wasting time on a full run.

3. **Run this notebook against the *preflight* alerts first.**  
   The first code cell is already wired to:

   - `ALERTS_PREFLIGHT_PATH = out/r5.2/alerts_preflight/alerts.ndjson` (short run)
   - `ALERTS_PATH          = out/r5.2/alerts_full/alerts.ndjson` (full run)

   For all the initial checks, **use `ALERTS_PREFLIGHT_PATH`** in the analysis cells.  
   Confirm:
   - `z_personal`, `z_role`, `z_max` are non-null for many anomaly alerts  
   - `anomaly_score_boosted` ≠ `anomaly_score_raw` for at least some alerts  
   - No silent failures in the z-score lookups or boosting logic

4. **Only switch to `ALERTS_PATH` / run the full `run_loop` once all checkpoints pass.**  
   If the preflight run looks clean, then (and only then) run the full loop:

       python -m src.run_loop --start 2010-01-02 --end 2011-06-02 --out-dir out/r5.2/alerts_full
---

> **Assumptions**
> - You are running this from the repo root via VS Code / Jupyter.
> - `release.txt` points at `r5.2`.
> - You already ran the full anomaly pipeline:
>   - `python -m src.anomaly.build_user_roles`
>   - `python -m src.anomaly.build_user_org_structure`
>   - `python -m src.anomaly.train_isolation_forest`
>   - `python -m src.anomaly.build_window_scores`
>   - `python -m src.anomaly.compute_baselines`

In [21]:
# Force notebook to use repo root so imports like `src.*` work
import sys
from pathlib import Path

# Walk upward until we find release.txt and src/
cwd = Path.cwd()
REPO_ROOT = cwd
for parent in [cwd] + list(cwd.parents):
    if (parent / "release.txt").exists() and (parent / "src").exists():
        REPO_ROOT = parent
        break

print("REPO_ROOT:", REPO_ROOT)

# Put repo root at front of import path
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("sys.path[0]:", sys.path[0])

REPO_ROOT: /Users/jordanchambers/capstone_6019
sys.path[0]: /Users/jordanchambers/capstone_6019


In [22]:
# Make sure notebook is using same interpreter & module
import sys, inspect, pathlib
import src.detector.anomaly as an

print("NB CWD:", pathlib.Path.cwd())
print("NB PYTHON:", sys.executable)
print("NB anomaly.py:", inspect.getsourcefile(an))

NB CWD: /Users/jordanchambers/capstone_6019/notebooks/ml
NB PYTHON: /Users/jordanchambers/capstone_6019/.venv/bin/python
NB anomaly.py: /Users/jordanchambers/capstone_6019/src/detector/anomaly.py


In [23]:
from pathlib import Path
import json

import pandas as pd
import numpy as np

# Resolve REPO_ROOT robustly by walking up until we find release.txt + out/
cwd = Path.cwd()
REPO_ROOT = cwd
for parent in [cwd] + list(cwd.parents):
    if (parent / "release.txt").exists() and (parent / "out").exists():
        REPO_ROOT = parent
        break

print("REPO_ROOT:", REPO_ROOT)

# Main alerts file from full run_loop
ALERTS_PATH = REPO_ROOT / "out" / "r5.2" / "alerts_full" / "alerts.ndjson"

# Optional smaller test file from a shorter run_loop range, if you want
ALERTS_PREFLIGHT_PATH = REPO_ROOT / "out" / "r5.2" / "alerts_preflight" / "alerts.ndjson"

ANSWERS_PATH = REPO_ROOT / "answers" / "insiders.csv"

DATASET_ID = "5.2"   # CERT dataset id as string in insiders.csv
SCENARIO_ID = 1      # Scenario 1 (USB + after-hours + Wikileaks)

print("ALERTS_PATH        :", ALERTS_PATH)
print("ALERTS_PREFLIGHT   :", ALERTS_PREFLIGHT_PATH)
print("ANSWERS_PATH       :", ANSWERS_PATH)

REPO_ROOT: /Users/jordanchambers/capstone_6019
ALERTS_PATH        : /Users/jordanchambers/capstone_6019/out/r5.2/alerts_full/alerts.ndjson
ALERTS_PREFLIGHT   : /Users/jordanchambers/capstone_6019/out/r5.2/alerts_preflight/alerts.ndjson
ANSWERS_PATH       : /Users/jordanchambers/capstone_6019/answers/insiders.csv


## 1. Ground truth: Scenario‑1 exfiltrators

Load the insider ground truth and keep only Scenario‑1 users for dataset r5.2. This is the set of users we **care most about** when evaluating anomaly behavior.

In [24]:
import duckdb

print("ANSWERS_PATH:", ANSWERS_PATH)
if not ANSWERS_PATH.exists():
    raise FileNotFoundError(f"{ANSWERS_PATH} does not exist. Fix ANSWERS_PATH above.")

# Pull Scenario-1 exfil ranges from insiders.csv
query = f"""
SELECT
  lower("user") AS user_key,
  strptime("start", '%m/%d/%Y %H:%M:%S') AS exfil_start,
  strptime("end",   '%m/%d/%Y %H:%M:%S') AS exfil_end
FROM read_csv_auto('{ANSWERS_PATH.as_posix()}', header = TRUE)
WHERE dataset = {DATASET_ID}
  AND scenario = {SCENARIO_ID}
ORDER BY exfil_start, user_key
"""

exfil_by_user = duckdb.sql(query).df()

# Debug guard: if we got zero rows, show what *does* exist
if exfil_by_user.empty:
    print(">>> No exfil rows matched DATASET_ID / SCENARIO_ID.")
    df_dist = duckdb.sql(f"""
        SELECT DISTINCT dataset, scenario
        FROM read_csv_auto('{ANSWERS_PATH.as_posix()}', header = TRUE)
        ORDER BY dataset, scenario
    """).df()
    print("Distinct (dataset, scenario) values in insiders.csv:")
    display(df_dist)
    raise RuntimeError(
        "No Scenario-1 exfil rows found. "
        "Check DATASET_ID / SCENARIO_ID constants in the first cell."
    )

# Normalize user_key to lowercase and parse dates
exfil_by_user["user_key"] = exfil_by_user["user_key"].str.lower()
exfil_by_user["exfil_start"] = pd.to_datetime(exfil_by_user["exfil_start"])
exfil_by_user["exfil_end"] = pd.to_datetime(exfil_by_user["exfil_end"])

print("=== exfil_by_user ===")
print(exfil_by_user.shape)
display(exfil_by_user.head(10))

ANSWERS_PATH: /Users/jordanchambers/capstone_6019/answers/insiders.csv
=== exfil_by_user ===
(29, 3)


,user_key,exfil_start,exfil_end
0,kew0198,2010-07-07 18:50:50,2010-07-14 06:27:14
1,das1320,2010-07-09 01:14:15,2010-07-15 01:38:58
2,epg1196,2010-07-15 03:00:42,2010-07-20 00:19:48
3,kbc1390,2010-07-16 02:18:07,2010-07-22 00:50:53
4,gfm1815,2010-07-16 22:00:57,2010-07-16 23:01:54
5,saf1942,2010-07-21 00:37:49,2010-07-23 02:12:05
6,pbc0077,2010-07-22 02:04:28,2010-07-31 02:28:03
7,alt1465,2010-08-13 22:16:10,2010-08-20 04:01:32
8,sll0193,2010-08-27 18:57:31,2010-09-03 00:24:08
9,ihc0561,2010-09-02 06:10:00,2010-09-10 08:28:40


## 2. Helper: load alerts NDJSON

This function loads NDJSON alerts into a DataFrame and does some light cleanup.

In [25]:
def load_alerts(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Alerts file not found: {path}")
    print(f"Loading alerts from: {path}")
    df = pd.read_json(path, lines=True)
    # Normalize day column to datetime, then back to YYYY-MM-DD string
    df["day"] = pd.to_datetime(df["day"]).dt.strftime("%Y-%m-%d")
    # Lowercase user_key for consistency
    df["user_key"] = df["user_key"].str.lower()
    return df

alerts = load_alerts(ALERTS_PATH)

print("=== alerts (all) ===")
print(alerts.shape)
print(alerts["detector"].value_counts())
alerts.head()

Loading alerts from: /Users/jordanchambers/capstone_6019/out/r5.2/alerts_full/alerts.ndjson
=== alerts (all) ===
(79175, 10)
detector
anomaly     75710
forecast     2201
rules         631
loop          517
ml            116
Name: count, dtype: int64


,day,user_key,detector,reason,score,evidence,human_summary,rules_score,anomaly_score,forecast_score
0,2010-01-02,aab1302,loop,heartbeat,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-03,acc0950,loop,heartbeat,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-04,emr0269,rules,rules:s1_near_miss,0.3,"{'ah_sig_days_7d': 2, 'usb_days_7d': 3, 'usb_t...",Possible preparation for data theft: after-hou...,NaN,NaN,NaN
3,2010-01-04,vrp0267,rules,rules:s1_near_miss,0.3,"{'ah_sig_days_7d': 2, 'usb_days_7d': 3, 'usb_t...",Possible preparation for data theft: after-hou...,NaN,NaN,NaN
4,2010-01-04,aab1302,loop,heartbeat,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Focused view: alerts for Scenario‑1 insiders only

Here we keep only alerts for the 29 Scenario‑1 users. This is the slice that matters most for lead‑time and detector usefulness.

In [26]:
s1_users = set(exfil_by_user["user_key"])
alerts_s1 = alerts[alerts["user_key"].isin(s1_users)].copy()

print("=== alerts_s1 (S1 exfil users only) ===")
print(alerts_s1.shape)
print(alerts_s1["detector"].value_counts())
alerts_s1.head()

=== alerts_s1 (S1 exfil users only) ===
(1045, 10)
detector
anomaly     593
forecast    225
ml          116
rules       111
Name: count, dtype: int64


,day,user_key,detector,reason,score,evidence,human_summary,rules_score,anomaly_score,forecast_score
2381,2010-01-22,dnj0740,forecast,forecast:s1_exfil_7d,0.537334,{},NaN,NaN,NaN,NaN
2693,2010-01-25,dnj0740,forecast,forecast:s1_exfil_7d,0.727472,{},NaN,NaN,NaN,NaN
2921,2010-01-26,dnj0740,forecast,forecast:s1_exfil_7d,0.699153,{},NaN,NaN,NaN,NaN
3260,2010-01-27,saf1942,forecast,forecast:s1_exfil_7d,0.753664,{},NaN,NaN,NaN,NaN
7591,2010-02-25,das1320,forecast,forecast:s1_exfil_7d,0.611604,{},NaN,NaN,NaN,NaN


## 4. Anomaly alerts: evidence unpacking

Pull out anomaly alerts and flatten the `evidence` field so we can inspect:

- `anomaly_score_raw`
- `anomaly_score_boosted`
- `boost_pct`
- `z_personal`, `z_role`, `z_max`
- `window_days`, `window_end`

In [27]:
anomaly_all = alerts[alerts["detector"] == "anomaly"].copy()
print("Total anomaly alerts (all users):", len(anomaly_all))

# Flatten evidence dict into columns
ev = pd.json_normalize(anomaly_all["evidence"].fillna({}))
ev_cols = ev.columns
print("Evidence columns:", list(ev_cols))

for col in ev_cols:
    anomaly_all[col] = ev[col]

# Sanity: check z-score non-null rate
z_cols = ["z_personal", "z_role", "z_max"]
for zc in z_cols:
    non_null = anomaly_all[zc].notna().sum()
    print(f"{zc}: {non_null} non-null out of {len(anomaly_all)}")

non_null_any = anomaly_all[z_cols].notna().any(axis=1).sum()
print(f"Rows with any non-null z_*: {non_null_any}")

# Check boosted vs raw and boost_pct
diff = (anomaly_all["anomaly_score_boosted"].round(4) != anomaly_all["anomaly_score_raw"].round(4))
boost_nonzero = anomaly_all["boost_pct"].fillna(0) != 0

print("Rows with boosted != raw:", int(diff.sum()))
print("Rows with boost_pct != 0:", int(boost_nonzero.sum()))

anomaly_all.head()

Total anomaly alerts (all users): 75710
Evidence columns: ['anomaly_score_raw', 'anomaly_score_boosted', 'boost_pct', 'base_score_raw_if', 'z_personal', 'z_role', 'z_max', 'window_days', 'window_end']
z_personal: 68550 non-null out of 75710
z_role: 70770 non-null out of 75710
z_max: 71034 non-null out of 75710
Rows with any non-null z_*: 71034
Rows with boosted != raw: 41166
Rows with boost_pct != 0: 37777


,day,user_key,detector,reason,score,evidence,human_summary,rules_score,anomaly_score,forecast_score,anomaly_score_raw,anomaly_score_boosted,boost_pct,base_score_raw_if,z_personal,z_role,z_max,window_days,window_end
78,2010-01-08,atd0517,anomaly,anomaly:isolation_forest,0.585823,"{'anomaly_score_raw': 0.5858, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5740,0.5740,0.0,0.0539,NaN,NaN,NaN,8.0,2010-01-11
85,2010-01-08,ddr1876,anomaly,anomaly:isolation_forest,0.521751,"{'anomaly_score_raw': 0.5218, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.6890,0.6890,0.0,0.1000,NaN,NaN,NaN,7.0,2010-01-11
87,2010-01-08,emr0269,anomaly,anomaly:isolation_forest,0.637295,"{'anomaly_score_raw': 0.6373, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5709,0.5709,0.0,0.0527,NaN,NaN,NaN,8.0,2010-01-11
91,2010-01-08,gat1343,anomaly,anomaly:isolation_forest,0.592649,"{'anomaly_score_raw': 0.5926, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.7216,0.7216,0.0,0.1130,NaN,NaN,NaN,8.0,2010-01-11
95,2010-01-08,hre1950,anomaly,anomaly:isolation_forest,0.642666,"{'anomaly_score_raw': 0.6427, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5050,0.5050,0.0,0.0263,NaN,NaN,NaN,7.0,2010-01-12


In [28]:
def summarize_threshold(df_all, s1_users, thr: float) -> dict:
    kept = df_all[df_all["anomaly_score_boosted"] >= thr]

    total_kept = len(kept)
    kept_s1 = kept[kept["user_key"].isin(s1_users)]
    s1_users_hit = kept_s1["user_key"].nunique()
    s1_alerts = len(kept_s1)

    non_s1_alerts = total_kept - s1_alerts
    non_s1_per_s1 = non_s1_alerts / s1_alerts if s1_alerts > 0 else float("inf")

    total_users_hit = kept["user_key"].nunique()
    user_precision = (
        s1_users_hit / total_users_hit if total_users_hit > 0 else float("nan")
    )

    n_s1_total = len(s1_users)
    s1_user_recall = s1_users_hit / n_s1_total if n_s1_total > 0 else float("nan")

    return {
        "threshold": thr,
        "alerts_kept": total_kept,
        "s1_users_hit": s1_users_hit,
        "s1_user_recall": round(s1_user_recall, 3),
        "s1_alerts": s1_alerts,
        "non_s1_per_s1_alert": round(non_s1_per_s1, 1) if total_kept > 0 else float("nan"),
        "user_precision": round(user_precision, 3),
    }


base_thr = 0.4   # current detector gate
new_thr  = 0.75  # candidate stricter gate

base_stats = summarize_threshold(anomaly_all, s1_users, base_thr)
new_stats  = summarize_threshold(anomaly_all, s1_users, new_thr)

pd.DataFrame([base_stats, new_stats])

,threshold,alerts_kept,s1_users_hit,s1_user_recall,s1_alerts,non_s1_per_s1_alert,user_precision
0,0.40,72321,29,1.0,585,122.6,0.035
1,0.75,12637,29,1.0,107,117.1,0.041


In [29]:
print("=== Anomaly z-score sanity check ===")
print("Total anomaly alerts:", len(anomaly_all))

z_cols = ["z_personal", "z_role", "z_max"]

null_counts = anomaly_all[z_cols].isna().sum()
null_frac   = (anomaly_all[z_cols].isna().mean() * 100).round(2)

print("\nNull counts:")
print(null_counts)

print("\nNull percentages:")
print(null_frac)

# Hard fail if everything is null
if (null_counts == len(anomaly_all)).all():
    raise RuntimeError("All z_* columns are NULL. Bug is NOT fixed. Do NOT trust anomaly outputs.")
elif (null_frac > 50).any():
    print("\nWARNING: More than 50% null in at least one z_* column. Something still smells off.")
else:
    print("\nOK: z_* columns are mostly non-null. Null-zscore bug is effectively fixed.")

=== Anomaly z-score sanity check ===
Total anomaly alerts: 75710

Null counts:
z_personal    7160
z_role        4940
z_max         4676
dtype: int64

Null percentages:
z_personal    9.46
z_role        6.52
z_max         6.18
dtype: float64

OK: z_* columns are mostly non-null. Null-zscore bug is effectively fixed.


### 4.1. Hard check: flag the **null z-score bug**

If the anomaly detector is silently ignoring z‑scores, **all** z_* will be null and this cell will raise loudly.

In [30]:
total_anom = len(anomaly_all)
rows_with_any_z = anomaly_all[z_cols].notna().any(axis=1).sum()

print(f"Total anomaly alerts     : {total_anom}")
print(f"Alerts with any non-null z: {rows_with_any_z}")

if total_anom > 0 and rows_with_any_z == 0:
    raise RuntimeError("BUG: anomaly alerts have all-null z_* values. Do NOT trust this run_loop output.")
else:
    print("OK: anomaly alerts have non-null z_* values.")

Total anomaly alerts     : 75710
Alerts with any non-null z: 71034
OK: anomaly alerts have non-null z_* values.


## 5. Anomaly alerts for Scenario‑1 insiders only

Now restrict to the 29 S1 users and see how many anomaly alerts hit them, and whether z‑scores look sane.

In [31]:
anomaly_s1 = anomaly_all[anomaly_all["user_key"].isin(s1_users)].copy()

print("=== anomaly_s1 (S1 insiders only) ===")
print(anomaly_s1.shape)
print(anomaly_s1[["user_key", "day"]].head())

z_non_null_s1 = anomaly_s1[z_cols].notna().any(axis=1).sum()
print("S1 anomaly alerts with any non-null z_*:", z_non_null_s1)

=== anomaly_s1 (S1 insiders only) ===
(593, 19)
      user_key         day
19075  das1320  2010-05-09
19146  das1320  2010-05-10
19365  das1320  2010-05-11
19580  das1320  2010-05-12
19795  das1320  2010-05-13
S1 anomaly alerts with any non-null z_*: 585


## 6. Threshold sweep to reduce anomaly spam

Goal: **pick an anomaly score threshold** that cuts down total alerts but still gives useful signal on S1 insiders.

We sweep boosted anomaly score thresholds and measure, for each threshold:

- total anomaly alerts kept
- number of S1 users with at least one alert
- fraction of all anomaly alerts that are S1
- basic z-score stats for the kept alerts

In [32]:
def sweep_anomaly_thresholds(
    df_all: pd.DataFrame,
    s1_users: set[str],
    thresholds=None
) -> pd.DataFrame:
    """
    For each anomaly threshold, compute:
      - how many alerts survive
      - how many of them belong to S1 users
      - how many UNIQUE S1 users are hit (recall)
      - how noisy things are (non-S1 alerts per S1 alert)
      - how concentrated on S1 users we are (user_precision)
      - how strong the z-max is on average
    """
    if thresholds is None:
        thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

    records = []
    n_s1_total = len(s1_users)

    for thr in thresholds:
        kept = df_all[df_all["anomaly_score_boosted"] >= thr]

        total_kept = len(kept)
        if total_kept == 0:
            records.append({
                "threshold": thr,
                "alerts_kept": 0,
                "s1_users_hit": 0,
                "s1_user_recall": 0.0,
                "s1_alerts": 0,
                "s1_fraction": 0.0,
                "non_s1_per_s1_alert": np.nan,
                "user_precision": np.nan,
                "mean_z_max": np.nan,
                "median_z_max": np.nan,
            })
            continue

        kept_s1 = kept[kept["user_key"].isin(s1_users)]
        s1_users_hit = kept_s1["user_key"].nunique()
        s1_alerts = len(kept_s1)
        s1_fraction = s1_alerts / total_kept

        # How many *non*-S1 alerts per S1 alert (noise level)
        non_s1_alerts = total_kept - s1_alerts
        non_s1_per_s1 = (
            non_s1_alerts / s1_alerts if s1_alerts > 0 else np.inf
        )

        # Among all users above threshold, what fraction are S1?
        total_users_hit = kept["user_key"].nunique()
        user_precision = (
            s1_users_hit / total_users_hit if total_users_hit > 0 else np.nan
        )

        # Recall over S1 users
        s1_user_recall = s1_users_hit / n_s1_total if n_s1_total > 0 else np.nan

        mean_z = kept["z_max"].mean()
        median_z = kept["z_max"].median()

        records.append({
            "threshold": thr,
            "alerts_kept": total_kept,
            "s1_users_hit": s1_users_hit,
            "s1_user_recall": round(s1_user_recall, 3),
            "s1_alerts": s1_alerts,
            "s1_fraction": round(s1_fraction, 3),
            "non_s1_per_s1_alert": round(non_s1_per_s1, 1) if np.isfinite(non_s1_per_s1) else np.inf,
            "user_precision": round(user_precision, 3),
            "mean_z_max": round(mean_z, 3) if pd.notna(mean_z) else np.nan,
            "median_z_max": round(median_z, 3) if pd.notna(median_z) else np.nan,
        })

    return pd.DataFrame.from_records(records)

sweep_df = sweep_anomaly_thresholds(
    anomaly_all,
    s1_users,
    thresholds=[0.3, 0.4, 0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9]
)
display(sweep_df)

,threshold,alerts_kept,s1_users_hit,s1_user_recall,s1_alerts,s1_fraction,non_s1_per_s1_alert,user_precision,mean_z_max,median_z_max
0,0.30,72321,29,1.000,585,0.008,122.6,0.035,2.820,2.54
1,0.40,72321,29,1.000,585,0.008,122.6,0.035,2.820,2.54
2,0.50,57322,29,1.000,476,0.008,119.4,0.035,2.759,2.61
3,0.60,32296,29,1.000,272,0.008,117.7,0.036,3.008,2.86
4,0.70,19635,29,1.000,166,0.008,117.3,0.038,3.205,3.02
5,0.75,12637,29,1.000,107,0.008,117.1,0.041,3.377,3.14
6,0.80,8387,26,0.897,76,0.009,109.4,0.041,3.529,3.22
7,0.85,5730,22,0.759,49,0.009,115.9,0.039,3.625,3.28
8,0.90,2889,16,0.552,25,0.009,114.6,0.035,3.923,3.52


In [33]:
min_scores = (
    anomaly_all[anomaly_all['user_key'].isin(s1_users)]
    .groupby('user_key')['anomaly_score_boosted']
    .min()
    .sort_values()
)
min_scores

user_key
ayg1697    0.4008
jup1472    0.4013
saf1942    0.4017
tmc0934    0.4029
whb1247    0.4030
alt1465    0.4044
das1320    0.4058
ihc0561    0.4067
www0701    0.4078
kbc1390    0.4111
sll0193    0.4131
niv1608    0.4134
wsk1857    0.4139
fzg0389    0.4152
etw0002    0.4179
gfm1815    0.4182
jkb0287    0.4219
zkp0542    0.4222
elt1370    0.4224
pbc0077    0.4255
ref1924    0.4268
alw0764    0.4307
dnj0740    0.4325
vah1292    0.4396
isw0738    0.4431
kew0198    0.4571
epg1196    0.4579
mib0203    0.4596
pth0005    0.4885
Name: anomaly_score_boosted, dtype: float64

### 6.1. How to read this table

- **alerts_kept**: total anomaly alerts you’d show to the UI / analysts.
- **s1_users_hit**: how many of the 29 S1 insiders get *any* anomaly alert at that threshold.
- **s1_fraction**: what portion of all anomaly alerts belong to S1 insiders.

The sweet spot is usually a threshold where:
- `s1_users_hit` is close to 29 (or at least doesn’t plummet), and
- `alerts_kept` is dramatically lower than the raw 50k+,
- `s1_fraction` increases compared to lower thresholds.

You can tweak the `thresholds` list in the cell above to zoom in (e.g., `[0.4, 0.45, 0.5, 0.55, 0.6]`).

## 7. Optional: sanity check on a smaller preflight alerts file

If you ran a shorter `run_loop` into `out/r5.2/alerts_preflight/alerts.ndjson`, this cell mirrors the z‑score bug check on that file.

In [34]:
if ALERTS_PREFLIGHT_PATH.exists():
    print("=== Preflight alerts z-score check ===")
    pre = load_alerts(ALERTS_PREFLIGHT_PATH)
    pre_anom = pre[pre["detector"] == "anomaly"].copy()
    if not pre_anom.empty:
        ev_pre = pd.json_normalize(pre_anom["evidence"].fillna({}))
        for col in ev_pre.columns:
            pre_anom[col] = ev_pre[col]
        z_non_null_pre = pre_anom[z_cols].notna().any(axis=1).sum()
        print("Preflight anomaly alerts:", len(pre_anom))
        print("Preflight alerts with any non-null z_*:", z_non_null_pre)
    else:
        print("No anomaly alerts in preflight file.")
else:
    print("No preflight alerts file found; skipping this check.")

=== Preflight alerts z-score check ===
Loading alerts from: /Users/jordanchambers/capstone_6019/out/r5.2/alerts_preflight/alerts.ndjson


KeyError: 'day'

In [35]:
display(sweep_df)

print("\nHeuristic recommendation:")
# Keep thresholds that hit all Scenario-1 users
full_recall = sweep_df[sweep_df["s1_users_hit"] == sweep_df["s1_users_hit"].max()]

# Among those, prefer the one that drops alerts_kept the most without being absurd
best_row = full_recall.sort_values("alerts_kept").iloc[0]
thr = best_row["threshold"]

print(f"- All {best_row['s1_users_hit']} S1 users are still covered at every tested threshold.")
print(f"- Smallest alerts_kept with full S1 coverage is threshold={thr:.1f}")
print(f"  (alerts_kept={int(best_row['alerts_kept'])}, s1_alerts={int(best_row['s1_alerts'])})")

,threshold,alerts_kept,s1_users_hit,s1_user_recall,s1_alerts,s1_fraction,non_s1_per_s1_alert,user_precision,mean_z_max,median_z_max
0,0.30,72321,29,1.000,585,0.008,122.6,0.035,2.820,2.54
1,0.40,72321,29,1.000,585,0.008,122.6,0.035,2.820,2.54
2,0.50,57322,29,1.000,476,0.008,119.4,0.035,2.759,2.61
3,0.60,32296,29,1.000,272,0.008,117.7,0.036,3.008,2.86
4,0.70,19635,29,1.000,166,0.008,117.3,0.038,3.205,3.02
5,0.75,12637,29,1.000,107,0.008,117.1,0.041,3.377,3.14
6,0.80,8387,26,0.897,76,0.009,109.4,0.041,3.529,3.22
7,0.85,5730,22,0.759,49,0.009,115.9,0.039,3.625,3.28
8,0.90,2889,16,0.552,25,0.009,114.6,0.035,3.923,3.52



Heuristic recommendation:
- All 29.0 S1 users are still covered at every tested threshold.
- Smallest alerts_kept with full S1 coverage is threshold=0.8
  (alerts_kept=12637, s1_alerts=107)


In [36]:
assert anomaly_all['z_max'].notna().sum() > 1000, \
    "Z-scores missing in full run. Stop everything."
assert (anomaly_all['anomaly_score_boosted'] != anomaly_all['anomaly_score_raw']).any(), \
    "No boosting detected in full alerts. Something is wrong."
print("Full-run anomaly integration looks correct.")

Full-run anomaly integration looks correct.


In [37]:
print("Total anomaly alerts:", len(anomaly_all))

print("\nNull rate per z-field:")
for col in ["z_personal", "z_role", "z_max"]:
    if col in anomaly_all.columns:
        null_rate = anomaly_all[col].isna().mean()
        print(f"{col}: {null_rate:.3f}")

print("\nExample rows with non-null z_max:")
display(anomaly_all[anomaly_all["z_max"].notna()].head(10))

Total anomaly alerts: 75710

Null rate per z-field:
z_personal: 0.095
z_role: 0.065
z_max: 0.062

Example rows with non-null z_max:


,day,user_key,detector,reason,score,evidence,human_summary,rules_score,anomaly_score,forecast_score,anomaly_score_raw,anomaly_score_boosted,boost_pct,base_score_raw_if,z_personal,z_role,z_max,window_days,window_end
1040,2010-01-15,bcg0853,anomaly,anomaly:isolation_forest,0.530892,"{'anomaly_score_raw': 0.5309, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5325,0.5325,0.0,0.0373,NaN,NaN,1.27,14.0,2010-01-17
1045,2010-01-15,bis1598,anomaly,anomaly:isolation_forest,0.710451,"{'anomaly_score_raw': 0.7105, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5143,0.5143,0.0,0.0300,NaN,NaN,1.37,14.0,2010-01-17
1047,2010-01-15,bne1140,anomaly,anomaly:isolation_forest,0.539973,"{'anomaly_score_raw': 0.54, 'anomaly_score_boo...",NaN,NaN,NaN,NaN,0.5909,0.5909,0.0,0.0607,NaN,NaN,1.71,14.0,2010-01-17
1048,2010-01-15,bwj1539,anomaly,anomaly:isolation_forest,0.611351,"{'anomaly_score_raw': 0.6114, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5866,0.5866,0.0,0.0590,NaN,NaN,1.33,14.0,2010-01-17
1049,2010-01-15,byo1846,anomaly,anomaly:isolation_forest,0.531939,"{'anomaly_score_raw': 0.5319, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.7354,0.7354,0.0,0.1186,NaN,NaN,2.34,14.0,2010-01-17
1050,2010-01-15,cac1083,anomaly,anomaly:isolation_forest,0.571816,"{'anomaly_score_raw': 0.5718, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.6476,0.6476,0.0,0.0834,NaN,NaN,1.38,14.0,2010-01-17
1051,2010-01-15,caf0109,anomaly,anomaly:isolation_forest,0.511617,"{'anomaly_score_raw': 0.5116, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.5997,0.5997,0.0,0.0642,NaN,NaN,1.82,14.0,2010-01-17
1054,2010-01-15,cas1068,anomaly,anomaly:isolation_forest,0.632618,"{'anomaly_score_raw': 0.6326, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.6064,0.6064,0.0,0.0669,NaN,NaN,1.35,14.0,2010-01-17
1056,2010-01-15,cbd1574,anomaly,anomaly:isolation_forest,0.541180,"{'anomaly_score_raw': 0.5412, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.6657,0.6657,0.0,0.0906,NaN,NaN,1.77,14.0,2010-01-17
1058,2010-01-15,ccb1836,anomaly,anomaly:isolation_forest,0.716490,"{'anomaly_score_raw': 0.7165, 'anomaly_score_b...",NaN,NaN,NaN,NaN,0.6035,0.6035,0.0,0.0657,NaN,NaN,2.14,14.0,2010-01-17


In [ ]:
print("Boost stats:")
print(anomaly_all["boost_pct"].value_counts(dropna=False).head(20))

print("\nBoosted examples (boost_pct > 0):")
display(anomaly_all[anomaly_all["boost_pct"] > 0].head(10))

print("\nDamped examples (boost_pct < 0):")
display(anomaly_all[anomaly_all["boost_pct"] < 0].head(10))